# CoC KeeperBot

---


In [ ]:
# verify library installs in your current python env
%pip install -r requirements.txt

## Setup & Initialization

### 1. Using gemini LLM model


In [3]:
import os
from dotenv import load_dotenv

loaded = load_dotenv()  # loads GEMINI_API_KEY

if not loaded:
    print("❌ Error: .env file not found!")
elif not os.getenv("GEMINI_API_KEY"):
    print("❌ Error: GEMINI_API_KEY not found in .env!")
else:
    print("✅ Environment loaded successfully. API Key detected.")

try:
    from langchain_google_genai import ChatGoogleGenerativeAI
    import chromadb

    print("✅ Libraries imported successfully.")
except ImportError as e:
    print(f"❌ Library Import Error: {e}")
    print("Did you run: pip install -r requirements.txt?")

✅ Environment loaded successfully. API Key detected.
✅ Libraries imported successfully.


In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", temperature=0.7
)  # Initialize the LLM
# More model options:https://ai.google.dev/gemini-api/docs/pricing

In [5]:
# Connection Test
try:
    print("Asking Gemini to introduce itself...")
    response = llm.invoke(
        "You are the Keeper of Arcane Lore in a Call of Cthulhu game. Introduce yourself in one sentence."
    )
    print(f"\nResponse:\n{response.content}")
except Exception as e:
    print(f"❌ API Call Failed: {e}")

Asking Gemini to introduce itself...

Response:
I am the boundless tome, the whispering archive, the keeper of all that should remain forgotten.


### Use the Qwen LLM model as a backup in case the Gemini API is unavailable.


In [ ]:
# pip install langchain-community dashscope
from dotenv import load_dotenv

loaded = load_dotenv()
if not loaded:
    print("❌ Error: .env file not found!")
elif not os.getenv("DASHSCOPE_API_KEY"):
    print("❌ Error: DASHSCOPE_API_KEY not found in .env!")
else:
    print("✅ Environment loaded successfully. API Key detected.")

#
from langchain_community.chat_models import ChatTongyi

llm = ChatTongyi(
    model="qwen-flash",  # support function calling model
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    temperature=0.7,
)


In [ ]:
# Connection Test
try:
    print("Asking Qwen to introduce itself...")
    response = llm.invoke(
        "You are the Keeper of Arcane Lore in a Call of Cthulhu game. Introduce yourself in one sentence."
    )
    print(f"\nResponse:\n{response.content}")
except Exception as e:
    print(f"❌ API Call Failed: {e}")

---

## Build the Knowledge Base (Run ONCE)


In [ ]:
# only needed if you are behind a proxy, e.g., China cannot access external APIs/Huggingface directly
%run config//jupyter_proxy.py

In [1]:
from src.rag_engine import build_vector_database, get_retriever, get_engine

# 1. Build the DB (set force_rebuild=True if the PDF or code is changed)
# Gemini embedding exceed the free tier limit
# NOTE: changed to HuggingFace Embeddings (Run loccally and no limits)
build_vector_database(reset=True)
# build_vector_database(reset=False)
engine = get_engine()
db = engine.db
print("count:", db._collection.count())

/opt/anaconda3/envs/ml4nlp-proj/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 Building RAG DB at ./data/chroma_db...


/opt/anaconda3/envs/ml4nlp-proj/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Ingestion done. DB now has 166 chunks.
✅ Database rebuilt successfully! chunks=166
count: 166


In [2]:
from src.rag_engine import build_vector_database, get_retriever, get_engine
# 2. Test Retrieval

return_raw_docs = False  # True

# Initialize RAG Engine
engine = get_engine()
db = engine.db
print("count:", db._collection.count())

# search_type = "similarity_score_threshold"
search_type = "mmr"
# Investigators can ask questions about rules only
test_retriever = "How to calculate SAN"
print(f"TEST: role: Investigators\n {test_retriever}")
hits = engine.retrieve(
    test_retriever,
    category="rule_system",
    use_rerank=True,
    search_type=search_type,
    return_raw_docs=return_raw_docs,
)
print("num docs:", len(hits))


for i, hit in enumerate(hits):
    print(f"\n--- Doc {i} ---")
    if return_raw_docs:
        print(hit.page_content)
    else:
        print(
            f"Source: {hit.get('source')}, Page: {hit.get('page')}, Chunk ID: {hit.get('chunk_id')}, Category: {hit.get('category')}, Rerank Score: {hit.get('rerank_score')}"
        )
        print(hit.get("text"))


# Keepers can ask any question about rules or the scenario
test_retriever = "Intrduce the scenario,The Haunting"
print(f"\n\n\nTEST: role: Keeper\n {test_retriever}")
hits = engine.retrieve(
    test_retriever,
    category=None,
    use_rerank=True,
    search_type=search_type,
    return_raw_docs=return_raw_docs,
)
print("num docs:", len(hits))
for i, hit in enumerate(hits):
    print(f"\n--- Doc {i} ---")
    if return_raw_docs:
        print(hit.page_content)
    else:
        print(
            f"Source: {hit.get('source')}, Page: {hit.get('page')}, Chunk ID: {hit.get('chunk_id')}, Category: {hit.get('category')}, Rerank Score: {hit.get('rerank_score')}"
        )
        print(hit.get("text"))

count: 166
TEST: role: Investigators
 How to calculate SAN


/opt/anaconda3/envs/ml4nlp-proj/lib/python3.12/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


num docs: 15

--- Doc 0 ---
Source: CoC Quick Start Rules, Page: 8, Chunk ID: 28, Category: rule_system, Rerank Score: -4.473779678344727
mind-bending horrors of the Cthulhu Mythos. Sometimes, 
when such things overcome your investigator, they lose Sanity 
points, reflecting the lingering effect of such trauma—see 
Sanity, page 12. Note that “Starting” SAN begins equal to 
POW, but may later rise or fall through play, and the “Insane” 
box is used to write in one-fifth of the “Starting” SAN value.
Running along the bottom of this section are Temporary 
Insanity, Indefinite Insanity, Major Wound, Unconscious, 
and Dying—these are checked when certain events happen 
during the game.

--- Doc 1 ---
Source: CoC Quick Start Rules, Page: 8, Chunk ID: 26, Category: rule_system, Rerank Score: -5.534359931945801
solving, and ability to make leaps of logic and intuition. 
• POW: a combination of force of will, spirit, and mental stability. 
In addition, there there are four key values for an inv

---